### Define variables
Initial gb and fasta files from entire project should be first defined.

In [1]:
workdir = "plastomes/table2asn/cp"
cc_gb = "plastomes/table2asn/cc/fixed_table2asn/output.gbf"
cc_fna = "plastomes/table2asn/fasta_manual_description/Crepis_callicephala.fasta"

cp_gb = "fixed_Crepis_purpurea.gb"
cp_fna = "plastomes/final/gb2sequin_out/c_purpurea_final/03/Crepis_purpurea.fsa"

### Check CDS of genes with known RNA editing sites

In [33]:
from Bio import SeqIO, SeqRecord

# output psbL data
gb_file = cp_gb # file with fixed tRNAs
target_gene = "psbL"

with open(gb_file) as handle:
    for record in SeqIO.parse(handle, "gb"):
        for feature in record.features:
            if feature.type in ["CDS", "gene"]:
                if feature.qualifiers.get('gene')[0] == target_gene:
                    print(feature)
                    cds_seq = feature.extract(record.seq)
                    #print(feature.qualifiers.get('gene')[0], cds_seq[:20])
                    start = feature.location.start - 1
                    end = start + 3
                    codon = record.seq[start:end].reverse_complement()  # 0-based
                    #print(f"codon: {codon}")

type: gene
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']

type: CDS
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']
    Key: product, Value: ['photosystem II subunit L']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['TTQSNPNEQNVELNRTSLYWGLLLIFVLAVLFSNYFFN']



The _**psbL**_ has no canonical codon in coding sequence start position.
Exception should be added and transcription output should be modified by changing the first AA letter to M

In [34]:
# output ndhD data
target_gene = "ndhD"

aa_shift = 6
bp_shift = aa_shift * 3

for record in SeqIO.parse(gb_file, "gb"):
    for feature in record.features:
        if feature.type == "CDS":
            gene_name = feature.qualifiers.get("gene", [""])[0]
            if gene_name == target_gene:
                print(feature)
                
                start = int(feature.location.start)
                end = int(feature.location.end)
                strand = feature.location.strand
                na = record.seq[start : end]

                if strand == 1:
                    position_shifted = start + bp_shift
                    na_shifted = record.seq[start + bp_shift : end]
                    ncbi_coord_str = f"{position_shifted + 1}..{end}"
                elif strand == -1:
                    position_shifted = end - bp_shift
                    na_shifted = record.seq[start : end - bp_shift]
                    ncbi_coord_str = f"{start + 1}..{position_shifted}"
                else:
                    print("Strand orientation unknown.")
                    continue
                
                for na_seq in [na, na_shifted]:
                    # convert to 5'-3' before translation
                    coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                    aa_seq = coding_seq.translate(to_stop=False)
                    if na_seq == na_shifted:
                        print(f"Translated sequence first symbols (shifted {aa_shift} aa): {aa_seq[0:15]}")
                    else:
                        print(f"Translated sequence first symbols: {aa_seq[0:21]}")
                    

                print(f"\ngene strand: ({'+' if strand == 1 else '-'})")
                print(f"Original gene coordinates in Biopython are [{start}:{end}]")
                print(f"Original GenBank coordinates: {start + 1}..{end}")
                print(f"Adjusted (start for (+) or end for (-) strand) position after shifting by {aa_shift} AA in Biopython is {position_shifted}")
                print(f"New GenBank coordinates after {aa_shift} aa shift: {ncbi_coord_str}")

type: CDS
location: [112421:113942](-)
qualifiers:
    Key: gene, Value: ['ndhD']
    Key: product, Value: ['NADH dehydrogenase subunit D']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR']

Translated sequence first symbols: VYLVFTTNKFPWLTIIVVLPI
Translated sequence first symbols (shifted 6 aa): TNKFPWLTIIVVLPI

gene strand: (-)
Original gene coordinates in Biopython are [112421:113942]
Original GenBank coordinates: 112422..113942
Adjusted (start for (+) or end for (-

As we see, in Crepis purpurea, the _**ndhD**_ has 6 additional amino acids in the start of its CDS sequence, but C. callicephala has not. It may represent the annotation error in C. purpurea and should be fixed by adjusting the gene boundaries.

It also has non-canonical codon at the coding sequence start position. The first AA should be changed to M after adding exception.

So there are two issues:
1. Start position needs adjustment.
2. CDS translation needs modifying by changing the first AA letter to M.

### Fixing non-canonical start codon-related issues via code
#### Granular approach
As we need changing two genes with different combination of things to change, the most preferred is a granular step-by-step approach, when we:
1. Adjust _**ndhD**_ coordinates in `CDS` and `gene` features.
2. Fix start position issues in `CDS` features of _**ndhD**_ and _**psbL**_.

In [ ]:
from Bio import SeqIO
from Bio.SeqFeature import FeatureLocation

# Configuration
input_gb = "your_annotation.gb"
output_gb = "ndhD_fixed.gb"
target_gene = "ndhD"
aa_shift = 6
bp_shift = aa_shift * 3

# Load all records into memory
records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        # Safely check for target gene
        gene_name = feature.qualifiers.get("gene", [""])[0]
        if feature.type in ["gene", "CDS"] and gene_name == target_gene:
            old_start = int(feature.location.start)
            old_end = int(feature.location.end)
            strand = int(feature.location.strand)

            # Calculate new boundaries
            if strand == -1:
                # Reverse strand: downstream shift reduces the end coordinate
                new_start = old_start
                new_end = old_end - bp_shift
            elif strand == 1:
                # Forward strand: downstream shift increases the start coordinate
                new_start = old_start + bp_shift
                new_end = old_end
            else:
                continue

            # Validate that new length maintains open reading frame
            if (new_end - new_start) % 3 != 0:
                raise ValueError(f"New length for {target_gene} is not a multiple of 3.")

            # Update feature location (0-based, half-open)
            feature.location = FeatureLocation(new_start, new_end, strand)
            print(f"[{feature.type}] Updated {target_gene}: {old_start}..{old_end} -> {new_start}..{new_end} (0-based)")

            # Apply RNA editing fix only to CDS features
            if feature.type == "CDS":
                # Extract new CDS in 5'->3' orientation and translate
                cds_seq = feature.extract(record.seq)
                protein_seq = str(cds_seq.translate(to_stop=False))

                # Ensure translation starts with M (reflects C-to-U RNA editing)
                if protein_seq[0] != "M":
                    protein_seq = "M" + protein_seq[1:]

                feature.qualifiers["translation"] = [protein_seq]

                # Build 1-based closed coordinates for NCBI /transl_except
                if strand == -1:
                    # Reverse strand: genomic start codon spans [new_end-3 : new_end]
                    # Convert to 1-based: (new_end-2)..new_end
                    except_1based = f"complement({new_end - 2}..{new_end})"
                else:
                    # Forward strand: genomic start codon spans [new_start : new_start+3]
                    # Convert to 1-based: (new_start+1)..(new_start+3)
                    except_1based = f"{new_start + 1}..{new_start + 3}"

                # Add NCBI-compliant qualifier
                feature.qualifiers["transl_except"] = [f"(pos:{except_1based},aa=M)"]
                print(f"  -> Added /transl_except=\"(pos:{except_1based},aa=M)\"")
                print(f"  -> Updated /translation (length: {len(protein_seq)} aa)")

# Write corrected file
SeqIO.write(records, output_gb, "gb")
print(f"\nSaved corrected GenBank file to: {output_gb}")

In [ ]:
from Bio import SeqIO

input_gb = "fixed_tRNAs.gb"      # Replace with your GenBank file
output_gb = "fixed_start_codons.gb"
target_genes = {"psbL", "ndhD"}

records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        if feature.type == "CDS" and feature.qualifiers.get("gene", [""])[0] in target_genes:
            gene = feature.qualifiers["gene"][0]
            loc = feature.location
            
            # 1. Extract genomic start codon & determine 1-based coordinates
            if loc.strand == -1:
                # Reverse strand: start codon is at the 3' end of the genomic interval
                genomic_codon = record.seq[loc.end-3:loc.end].reverse_complement()
                start_codon_pos = f"complement({loc.end-2}..{loc.end})"
            else:
                # Forward strand: start codon is at the 5' end
                genomic_codon = record.seq[loc.start:loc.start+3]
                start_codon_pos = f"{loc.start+1}..{loc.start+3}"
                
            genomic_codon_str = str(genomic_codon).upper()
            
            # Skip if already standard ATG
            if genomic_codon_str == "ATG":
                print(f"✅ {gene}: Already has standard ATG start. No change needed.")
                continue
                
            # 2. Update /translation to start with M (edited protein)
            if "translation" in feature.qualifiers:
                current_trans = feature.qualifiers["translation"][0]
                if current_trans[0] != "M":
                    feature.qualifiers["translation"][0] = "M" + current_trans[1:]
                    print(f"🔄 {gene}: Updated translation to start with M.")
                    
            # 3. Add /transl_except qualifier (NCBI official format)
            except_val = f"(pos:{start_codon_pos},aa=M)"
            feature.qualifiers["transl_except"] = [except_val]
            print(f"📝 {gene}: Added /transl_except=\"{except_val}\"")
            
            # 4. Add clarifying note (optional but recommended for reviewers)
            if "note" not in feature.qualifiers:
                feature.qualifiers["note"] = []
            feature.qualifiers["note"].append("Start codon created by RNA editing (C-to-U)")

SeqIO.write(records, output_gb, "gb")
print(f"\n💾 Saved corrected GenBank to: {output_gb}")